In [1]:
from torchvision import datasets,transforms
from torch.utils.data import DataLoader

In [2]:
transform=transforms.ToTensor() # convert image into tensor 
train_data=datasets.MNIST(
    root='data',
    train=True,
    download=True,
    transform=transform
)

test_data=datasets.MNIST(
    root='data',
    train=False,
    download=True,
    transform=transform
)

In [3]:
train_loader=DataLoader(train_data,batch_size=32)
test_loader=DataLoader(test_data,batch_size=32)

## CNN

In [4]:
import torch.nn as  nn
import torch.optim as optim

In [5]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        self.conv_layer=nn.Sequential(
            # frist layer
            nn.Conv2d(in_channels=1,out_channels=32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
             # second layer
            nn.Conv2d(in_channels=32,out_channels=64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
        self.fc_layers=nn.Sequential(
            nn.Linear(64*7*7,256),
            nn.ReLU(),
            nn.Linear(256,10) 
        )
    def forward(self,x):
            x=self.conv_layer(x)
            x=x.view(x.size(0),-1)
            x=self.fc_layers(x)
            return x

In [6]:
model=CNN()
critenion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

#### Traninig the CNN

In [7]:
epochs=10
for epoch in range(epochs):
    epoch_training_loss=0.0
    for image ,labels in train_loader:
        optimizer.zero_grad()
        output=model.forward(image)
        loss=critenion(output,labels)
        loss.backward()
        optimizer.step()
        epoch_training_loss+=loss.item()
    print(f"epoch={epoch+1} and loss={epoch_training_loss/len(train_loader)}")
    

epoch=1 and loss=0.13847357238132196
epoch=2 and loss=0.04346395791375932
epoch=3 and loss=0.02648848971751116
epoch=4 and loss=0.018838512757592192
epoch=5 and loss=0.014114560466866453
epoch=6 and loss=0.010740266765415386
epoch=7 and loss=0.00902747956407933
epoch=8 and loss=0.007040410901523713
epoch=9 and loss=0.006588973156550994
epoch=10 and loss=0.006072677321701374


### Evaluation of Model

In [8]:
import torch

correct_labels = 0
total_labels = 0

all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)   
        
        _, predicted = torch.max(outputs, 1)
        
        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)   
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
accuracy = (correct_labels / total_labels) * 100
print(f"Accuracy = {accuracy:.2f}%")

Accuracy = 98.90%


In [9]:
from sklearn.metrics import confusion_matrix,classification_report

In [10]:
cm = confusion_matrix(all_labels, all_preds)
print(cm)

[[ 979    0    0    0    0    0    1    0    0    0]
 [   0 1132    0    0    0    2    0    1    0    0]
 [   0    0 1024    0    1    0    0    7    0    0]
 [   0    0    0  993    1   14    0    1    0    1]
 [   0    0    1    0  965    0    5    0    2    9]
 [   0    0    0    2    0  888    1    0    0    1]
 [   4    2    0    0    1   14  936    0    1    0]
 [   0    1    3    0    2    0    0 1007    1   14]
 [   1    1    1    2    0    2    0    0  964    3]
 [   0    0    0    0    2    2    0    2    1 1002]]


In [11]:
cm=classification_report(all_labels,all_preds)
print(cm)

              precision    recall  f1-score   support

           0       0.99      1.00      1.00       980
           1       1.00      1.00      1.00      1135
           2       1.00      0.99      0.99      1032
           3       1.00      0.98      0.99      1010
           4       0.99      0.98      0.99       982
           5       0.96      1.00      0.98       892
           6       0.99      0.98      0.98       958
           7       0.99      0.98      0.98      1028
           8       0.99      0.99      0.99       974
           9       0.97      0.99      0.98      1009

    accuracy                           0.99     10000
   macro avg       0.99      0.99      0.99     10000
weighted avg       0.99      0.99      0.99     10000



## RNN

In [12]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layer=1):
        super().__init__()
        self.hidden_size=hidden_size
        self.num_layer=num_layer
        #Rnn Layer
        self.rnn=nn.RNN(input_size,hidden_size,num_layer,batch_first=True)

        #fully connected layer
        self.fc=nn.Linear(hidden_size,10)

    def forward(self,x):
        h0=torch.zeros(self.num_layer,x.size(0),self.hidden_size)
        out,_=self.rnn(x,h0)
        out=self.fc(out[:,-1,:])
        return out
    

In [13]:
image,label=train_data[0]
input_size=image.shape[1]

In [14]:
model=RNN(input_size)
critersion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

### Training RNN 

In [15]:
epochs=10
for epoch in range(epochs):
    model.train()
    for xb,yb in train_loader:
        optimizer.zero_grad()
        xb=xb.squeeze(1)
        
        output=model(xb).squeeze()
        output=torch.sigmoid(output)
        loss=critersion(output,yb)
        loss.backward()
        optimizer.step()
    print(f"{epoch+1}/{epochs} amd loss={loss.item()}")

1/10 amd loss=1.7675114870071411
2/10 amd loss=1.7413777112960815
3/10 amd loss=1.6095428466796875
4/10 amd loss=1.5829497575759888
5/10 amd loss=1.5362178087234497
6/10 amd loss=1.4873638153076172
7/10 amd loss=1.4955042600631714
8/10 amd loss=1.505966067314148
9/10 amd loss=1.4832193851470947
10/10 amd loss=1.47593092918396


### Evaluation

In [23]:
from sklearn.metrics import classification_report

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    correct_vals = 0
    total_val = 0

    for xb, yb in test_loader:
        xb = xb.squeeze(1)

        outputs = model(xb)
        _, predicated = torch.max(outputs, 1)

        all_preds.extend(predicated.cpu().numpy())
        all_labels.extend(yb.cpu().numpy())

        total_val += yb.size(0)
        correct_vals += (predicated == yb).sum().item()

print(f"Accuracy = {correct_vals / total_val * 100:.2f}%")

# Final check
print(len(all_preds), len(all_labels))

print(classification_report(all_labels, all_preds))

Accuracy = 91.89%
10000 10000
              precision    recall  f1-score   support

           0       0.94      0.96      0.95       980
           1       0.96      0.98      0.97      1135
           2       0.96      0.93      0.95      1032
           3       0.91      0.90      0.90      1010
           4       0.93      0.90      0.92       982
           5       0.86      0.79      0.82       892
           6       0.90      0.92      0.91       958
           7       0.95      0.93      0.94      1028
           8       0.88      0.95      0.92       974
           9       0.88      0.90      0.89      1009

    accuracy                           0.92     10000
   macro avg       0.92      0.92      0.92     10000
weighted avg       0.92      0.92      0.92     10000



## 🔍 Model Comparison: RNN vs CNN

| Feature                     | RNN Model                     | CNN Model                     |
|----------------------------|------------------------------|------------------------------|
| Accuracy                   | 91.89%                       | 98.90% 🚀                    |
| Data Handling              | Sequential (row-wise image)  | Spatial (2D image)           |
| Feature Learning           | Temporal patterns            | Spatial features (edges, shapes) |
| Performance on MNIST       | Moderate                     | Excellent                    |
| Weak Classes               | 5, 3, 9                      | Slight confusion in 5, 9     |
| Training Speed             | Slower                       | Faster                       |
| Suitability for Images     | ❌ Not ideal                 | ✅ Best choice               |
| Key Limitation             | Loses spatial information    | Computationally heavier      |
| Overall Verdict            | Baseline model               | Production-ready model ✅     |

## 📈 Performance Summary

| Metric        | RNN Model | CNN Model |
|--------------|----------|----------|
| Accuracy     | 91.89%   | 98.90%   |
| Precision    | ~0.92    | ~0.99    |
| Recall       | ~0.92    | ~0.99    |
| F1-Score     | ~0.92    | ~0.99    |

## 🧠 Key Observations

| Observation |
|------------|
| CNN significantly outperforms RNN by ~7% accuracy |
| RNN struggles with spatial understanding of images |
| CNN effectively captures edges, shapes, and patterns |
| Digit '5' was hardest for both models |
| CNN predictions are more consistent across all classes |

## ✅ Conclusion

| Model | Conclusion |
|------|-----------|
| RNN  | Useful for sequence data but not ideal for images |
| CNN  | Best suited for image classification tasks like MNIST |

### 🚀 Final Insight

CNN achieved ~7% higher accuracy than RNN, proving that spatial feature extraction is crucial for image classification tasks.